### Trabajo Integrador Seminario de Actualización 2do Cuatrimestre 2026

#### Alumno: Angel Sebastián Medina
#### Profesora: Cynthia Villagra

#### Situación: vendedor en tienda de mascotas tiene problemas por reseñas en idioma inglés y tiene problemas para responderlas.
#### Solución: crear una aplicación cuya función sea traducir las reseñas desde el idioma inglés al español y viceversa para poder responder a ellas.

### Desarrollo de la aplicación

#### 1. Instalación de librerías necesarias

Para utilizar **pipeline**: *sentencepiece* y *sacremoses* son necesarias para el modelo de traducción. *torch* y *transformers* para **pipeline**.

***Primer problema***: Transformers debe ser anterior a v5 ya que versiones posteriores reemplazan la funcionalidad de traductor ("task=translator") por chats LLM.

In [4]:
%pip install "transformers<5" torch sentencepiece sacremoses

  Using cached transformers-4.57.6-py3-none-any.whl.metadata (43 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
Using cached transformers-4.57.6-py3-none-any.whl (12.0 MB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl (2.7 MB)
   ---------------------------------------- 0.0/867.8 kB ? eta -:--:--
   ---------------------------------------- 867.8/867.8 kB 15.3 MB/s  0:00:00

   ------ --------------------------------- 1/6 [joblib]
   ------ --------------------------------- 1/6 [joblib]
   ------ --------------------------------- 1/6 [joblib]
   ------ --------------------------------- 1/6 [joblib]
   ------ --------------------------------- 1/6 [joblib]
   ------ --------------------------------- 1/6 [joblib]
   ------ --------------------------------- 1/6 [joblib]
   ------ --------------------------------- 1/6 [

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.28.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Para utilizar **InferenceCliente**: Instalar el Hub de HuggingFace en caso de ser necesario.

In [ ]:
%pip install -q huggingface_hub

#### 2. Evaluación de los modelos con InferenceClient

Importamos **InferenceClient** y guardamos nuestro token de HuggingFace:

In [9]:
from huggingface_hub import InferenceClient

In [10]:
import os
from getpass import getpass

if "HF_TOKEN" not in os.environ:
    os.environ["HF_TOKEN"] = getpass("Pegá tu token de Hugging Face (no se va a ver): ")

Creamos la clase *traductor* a partir de la clase **InferenceClient**:

In [11]:
traductor = InferenceClient(provider="hf-inference", api_key=os.environ["HF_TOKEN"])
print(type(traductor))

<class 'huggingface_hub.inference._client.InferenceClient'>


Iniciamos la evaluación con el modelo específico Español -> Inglés con una frase al azar y obtenemos un error.

***Segundo Problema***: actualmente se posee una cuenta gratuita en HuggingFace, la cual brinda créditos gratuitos limitados, cuestión que puede llegar a ser un problema para el desempeño de la aplicación en el futuro.

In [ ]:
texto_a_traducir = "Estoy construyendo mi primer aplicación de traducción. Estoy muy emocionado."
resultado = traductor.translation(
    text=texto_a_traducir,
    model="Helsinki-NLP/opus-mt-es-en"
)

print(resultado.translation_text)

HfHubHTTPError: 402 Client Error: Payment Required for url: https://router.huggingface.co/hf-inference/models/Helsinki-NLP/opus-mt-es-en (Request ID: Root=1-6ac7a50a-2f01043c798cf90d5b0dc9e4;67071320-0d73-458f-9381-910b704c1c02)

You have no remaining credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get monthly included credits.

Intentaremos crear nuestra aplicación utilizando **pipeline**, descargando nuestro modelo en la pc.

#### 3. Evaluación de modelos utilizando librería Transformers

*Helsinki-NLP/opus-mt-es-en* y *Helsinki-NLP/opus-mt-en-es* son modelos con pares de idiomas específicos, pero más livianos para descargar

*facebook/mbart-large-50-many-to-many-mmt* es multilenguaje, pero es más pesado, más lento en la respuesta y con mayor cantidad de parámetros de configuración (tanto en Backend como en Frontend).

a. Traducción de ES (español) -> EN (inglés):

Modelo *Helsinki-NLP/opus-mt-es-en*. 

Al evaluar con la primer frase obtengo una sola oración traducida de las dos que utilizo.

In [1]:
from transformers import pipeline

c:\Users\asmed\Documents\Técnicatura Ciencia de Datos e IA\Seminario de Actualización\proyecto_integrador\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [20]:
traductor = pipeline("translation", model="Helsinki-NLP/opus-mt-es-en")
traduccion = traductor("Estoy construyendo mi primer aplicación de traducción. Estoy muy emocionado.")[0]["translation_text"]
print(f"Frase traducida: {traduccion}")
print(f"Función pipeline: {traductor.task}")
print(f"Tipo de dato de la traducción: {type(traduccion)}")

Device set to use cpu


Frase traducida: I'm building my first translation app.
Función pipeline: translation
Tipo de dato de la traducción: <class 'str'>


Pruebo con más oraciones en la frase y curiosamente funciona bien.

In [21]:
traductor = pipeline("translation", model="Helsinki-NLP/opus-mt-es-en")
traduccion = traductor("Estoy construyendo mi primer aplicación de traducción. Estoy muy emocionado. Espero que funcione bien.")[0]["translation_text"]
print(f"Frase traducida: {traduccion}")
print(f"Función pipeline: {traductor.task}")
print(f"Tipo de dato de la traducción: {type(traduccion)}")

Device set to use cpu


Frase traducida: I'm building my first translation app. I'm very excited. I hope it works well.
Función pipeline: translation
Tipo de dato de la traducción: <class 'str'>


Ahora una frase que podría responder el usuario de la aplicación:

In [22]:
traductor = pipeline("translation", model="Helsinki-NLP/opus-mt-en-es")
traduccion = traductor("Lamentamos lo sucedido. Su queja ha sido registrada. Mejoraremos nuestra atención.")[0]["translation_text"]
print(f"Frase traducida: {traduccion}")
print(f"Función pipeline: {traductor.task}")
print(f"Tipo de dato de la traducción: {type(traduccion)}")

Device set to use cpu


Frase traducida: Lamentamos lo sucedido. Su queja ha sido registrada. Mejoraremos nuestra atención.
Función pipeline: translation
Tipo de dato de la traducción: <class 'str'>


Al evaluar con una frase en otro idioma, como el italiano, el modelo no traduce correctamente, lo cual es esperable.

***Segundo problema***: El modelo está entrenado específicamente para traducir del español al inglés, y no tiene la capacidad de manejar otros idiomas. Por lo tanto, al utilizarlo es importante asegurar que el usuario ingrese de manera correcta el texto de entrada para obtener una traducción precisa.

In [23]:
traductor = pipeline("translation", model="Helsinki-NLP/opus-mt-es-en")
traduccion = traductor("È una giornata triste.")[0]["translation_text"]
print(f"Frase traducida: {traduccion}")
print(f"Función pipeline: {traductor.task}")
print(f"Tipo de dato de la traducción: {type(traduccion)}")

Device set to use cpu


Frase traducida: He's a sad giornata.
Función pipeline: translation
Tipo de dato de la traducción: <class 'str'>


Modelo *facebook/mbart-large-50-many-to-many-mmt*

En éste caso, debo detallar en el parámetro *task* el idioma origen y el resultante. A diferencia del primer modelo, funciona correctamente con dos oraciones de la primer frase:

In [24]:
traductor = pipeline("translation_ES_to_EN", model="facebook/mbart-large-50-many-to-many-mmt")
traduccion = traductor("Estoy construyendo mi primer aplicación de traducción. Estoy muy emocionado.")[0]["translation_text"]
print(f"Frase traducida: {traduccion}")
print(f"Función pipeline: {traductor.task}")
print(f"Tipo de dato de la traducción: {type(traduccion)}")

Device set to use cpu


Frase traducida: EN: I'm building my first translation application. I'm very excited.
Función pipeline: translation_ES_to_EN
Tipo de dato de la traducción: <class 'str'>


Evaluo con más oraciones en la frase obteniendo el mismo resultado.

In [25]:
traductor = pipeline("translation_ES_to_EN", model="facebook/mbart-large-50-many-to-many-mmt")
traduccion = traductor("Estoy construyendo mi primer aplicación de traducción. Estoy muy emocionado. Espero que funcione bien.")[0]["translation_text"]
print(f"Frase traducida: {traduccion}")
print(f"Función pipeline: {traductor.task}")
print(f"Tipo de dato de la traducción: {type(traduccion)}")

Device set to use cpu


Frase traducida: EN I'm building my first translation application. I'm very excited. I hope it works well.
Función pipeline: translation_ES_to_EN
Tipo de dato de la traducción: <class 'str'>


Al utilizar una frase con contexto, observamos que traduce con algunos errores:

In [26]:
traductor = pipeline("translation_ES_to_EN", model="facebook/mbart-large-50-many-to-many-mmt")
traduccion = traductor("Lamentamos lo sucedido. Su queja ha sido registrada. Mejoraremos nuestra atención.")[0]["translation_text"]
print(f"Frase traducida: {traduccion}")
print(f"Función pipeline: {traductor.task}")
print(f"Tipo de dato de la traducción: {type(traduccion)}")

Device set to use cpu


Frase traducida: EN: We're sorry about what happened. Your queen has been registered.
Función pipeline: translation_ES_to_EN
Tipo de dato de la traducción: <class 'str'>


Evaluación en italiano. Se obtiene un buen resultado, debido a que el modelo es multilenguaje, pudiendo identificar el idioma de origen correcto a pesar de haberlo configurado en español.

In [27]:
traductor = pipeline("translation_ES_to_EN", model="facebook/mbart-large-50-many-to-many-mmt")
traduccion = traductor("È una giornata triste.")[0]["translation_text"]
print(f"Frase traducida: {traduccion}")
print(f"Función pipeline: {traductor.task}")
print(f"Tipo de dato de la traducción: {type(traduccion)}")

Device set to use cpu


Frase traducida: EN: It's a sad day.
Función pipeline: translation_ES_to_EN
Tipo de dato de la traducción: <class 'str'>


b. Traducción de EN -> ES:

Modelo *Helsinki-NLP/opus-mt-en-es*

Cuando evaluo una frase desde el idioma correcto, el modelo responde bien.

In [29]:
traductor = pipeline("translation_EN_to_ES", model="Helsinki-NLP/opus-mt-en-es")
traduccion = traductor("My dog was very happy with the food the petshoper sold me.")[0]["translation_text"]
print(f"Frase traducida: {traduccion}")
print(f"Función pipeline: {traductor.task}")
print(f"Tipo de dato de la traducción: {type(traduccion)}")

Device set to use cpu


Frase traducida: Mi perro estaba muy contento con la comida que el vendedor de mascotas me vendió.
Función pipeline: translation_EN_to_ES
Tipo de dato de la traducción: <class 'str'>


Utilizamos una frase un poco más compleja, obteniendo buenos resultados:

In [30]:
traductor = pipeline("translation_EN_to_ES", model="Helsinki-NLP/opus-mt-en-es")
traduccion = traductor("My dog was very happy with the food the petshoper sold me. Nice attention!")[0]["translation_text"]
print(f"Frase traducida: {traduccion}")
print(f"Función pipeline: {traductor.task}")
print(f"Tipo de dato de la traducción: {type(traduccion)}")

Device set to use cpu


Frase traducida: Mi perro estaba muy contento con la comida que el vendedor de mascotas me vendió. ¡Buena atención!
Función pipeline: translation_EN_to_ES
Tipo de dato de la traducción: <class 'str'>


Utilizando la frase en italiano, el modelo no la reconoce como inglesa y no la traduce

In [31]:
traductor = pipeline("translation_EN_to_ES", model="Helsinki-NLP/opus-mt-en-es")
traduccion = traductor("È una giornata triste.")[0]["translation_text"]
print(f"Frase traducida: {traduccion}")
print(f"Función pipeline: {traductor.task}")
print(f"Tipo de dato de la traducción: {type(traduccion)}")

Device set to use cpu


Frase traducida: È una giornata triste.
Función pipeline: translation_EN_to_ES
Tipo de dato de la traducción: <class 'str'>


Modelo *facebook/mbart-large-50-many-to-many-mmt*

Vemos que al evaluar el modelo desde el inglés hacia el español, con ambas frases, comienza a fallar en su respuesta:

In [4]:
traductor = pipeline("translation_EN_to_ES", model="facebook/mbart-large-50-many-to-many-mmt")
traduccion = traductor("My dog was very happy with the food the petshoper sold me.")[0]["translation_text"]
print(f"Frase traducida: {traduccion}")
print(f"Función pipeline: {traductor.task}")
print(f"Tipo de dato de la traducción: {type(traduccion)}")

Device set to use cpu


Frase traducida: ES was very happy with the food the petshoper sold me My dogEN was very happy with the food the petshoper sold me.
Función pipeline: translation_EN_to_ES
Tipo de dato de la traducción: <class 'str'>


In [33]:
traductor = pipeline("translation_EN_to_ES", model="facebook/mbart-large-50-many-to-many-mmt")
traduccion = traductor("È una giornata triste.")[0]["translation_text"]
print(f"Frase traducida: {traduccion}")
print(f"Función pipeline: {traductor.task}")
print(f"Tipo de dato de la traducción: {type(traduccion)}")

Device set to use cpu


Frase traducida: ES a sad day.
Función pipeline: translation_EN_to_ES
Tipo de dato de la traducción: <class 'str'>


***Tercer Problema***: *pipeline* funciona correctamente para modelos de traducción directa entre dos idiomas. Para modelos multilenguaje como **MBart**, hay que dejar de lado **pipeline** y utilizar métodos auxiliares como *MBartForConditionalGeneration* y *MBart50TokenizerFast* para poder definir los *tokens* o códigos de idioma de origen y destino. Ésto complejiza el código, lo cual a pesar de no ser incorrecto, nos aleja de la consigna.

In [35]:
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast

nombre_modelo = "facebook/mbart-large-50-many-to-many-mmt"

# Indicamos idiomas de origen y destino al tokenizador
tokenizador = MBart50TokenizerFast.from_pretrained(
    nombre_modelo,
    src_lang="en_XX",   # inglés
    tgt_lang="es_XX"    # español
)
modelo = MBartForConditionalGeneration.from_pretrained(nombre_modelo)

texto = "My dog was very happy with the food the pet shop sold me."

inputs = tokenizador(texto, return_tensors="pt")

# Definimos la generación arranque con el token de español
output_ids = modelo.generate(
    **inputs,
    forced_bos_token_id=tokenizador.lang_code_to_id["es_XX"]
)

traduccion = tokenizador.batch_decode(output_ids, skip_special_tokens=True)[0]
print(traduccion)

print(f"Frase traducida: {traduccion}")
print(f"Función pipeline: {traductor.task}")
print(f"Tipo de dato de la traducción: {type(traduccion)}")

Mi perro estaba muy satisfecho con la comida que me vendió la tienda de mascotas.
Frase traducida: Mi perro estaba muy satisfecho con la comida que me vendió la tienda de mascotas.
Función pipeline: translation_EN_to_ES
Tipo de dato de la traducción: <class 'str'>


c. Tabla de Resultados:

Luego de la evaluación de los 3 modelos, observamos que con el par **opus** se obtienen mejores resultados en traducción, por lo que son la mejor opción para la aplicación. 

| Frase | opus/es-en | opus/en-es | Mbart/es-en | MBart/en-es |
| --- | --- | --- | --- | --- |
| Frase al azar | Traduce  | Traduce  | Traduce  | No evaluado |
| Frase con contexto | Traduce  | Traduce  | Traduce con error (no esperable) | Traduce con error |
| Frase en idioma incorrecto | No Traduce (esperable) | No Traduce (esperable)  | Traduce (no esperable) | Traduce con error |

#### 4. Prototipo de aplicación

Elegidos los modelos, vamos a crear un pequeño prototipo de la aplicación. La finalidad es observar más a fondo el desempeño de los modelos y las primeras funcionalidad del traductor:

In [1]:
from transformers import pipeline

# Modelos de traducción:
MODELOS = {
    "es-en": "Helsinki-NLP/opus-mt-es-en",
    "en-es": "Helsinki-NLP/opus-mt-en-es",
}

# Función para crear el pipeline de traducción:
def crear_traductor(par: str):
    """Crea y devuelve un pipeline de traducción para el par de idiomas elegido."""
    if par not in MODELOS:
        raise ValueError(
            f"Par '{par}' no soportado. Opciones: {', '.join(MODELOS)}"
        )
    
    return pipeline(
        task="translation",
        model=MODELOS[par],
    )

# Función para traducir un texto:
def traducir_texto(traductor, texto: str, max_length: int = 200) -> str:
    resultado = traductor(texto, max_length=max_length)
    return resultado[0]["translation_text"]

# Función principal:
def main():
    print("=== Traductor interactivo ===")
    print("Pares disponibles:", " | ".join(MODELOS))
    par = input("Elegí un par (ej: es-en): ").strip()
    traductor = crear_traductor(par)
    print(f"Traductor {par} listo. Escribí 'salir' para terminar.\n")
    while True:
        texto = input(">>> ").strip()
        if texto.lower() in ("salir", "exit", "quit"):
            break
        if not texto:
            continue
        print(traducir_texto(traductor, texto), "\n")

if __name__ == "__main__":
    main()

c:\Users\asmed\Documents\Técnicatura Ciencia de Datos e IA\Seminario de Actualización\proyecto_integrador\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== Traductor interactivo ===
Pares disponibles: es-en | en-es


Device set to use cpu


Traductor es-en listo. Escribí 'salir' para terminar.

Thank you so much for your visit! 

We appreciate your comment. We will take it into account! 



Observamos que en el prototipo los modelos infieren y funcionan correctamente. El paso siguiente será construir la interfaz gráfica con Gradio de la aplicación en un archivo **app.py**. El mismo se alojará en el siguiente repositorio de GitHub: https://github.com/sebsmedina/seminario_2026_proyecto_integrador/tree/main/proyecto_integrador